In [7]:
# Purpose: Set up imports/paths/device and discover all checkpoints + test images.

import os, glob
from pathlib import Path
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# ---- PATHS ----
MODELS_DIR = '/zpool/vladlab/active_drive/omaltz/scripts/geogaze/v3/resnet100/weights'
TEST_DIR   = '/zpool/vladlab/active_drive/omaltz/scripts/geogaze/stimuli/out/test_stimuli/test_pairs'
OUT_ROOT   = '/zpool/vladlab/active_drive/omaltz/scripts/geogaze/v3/resnet100/test_masks' 

# ---- INFERENCE SETTINGS ----
ARCH       = 'resnet50.a1_in1k'
THRESHOLD  = 0.3
MAX_PREVIEW = None 

# ---- DEVICE + NORM ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)  # keep on CPU; we'll move the batch later
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

# ---- DISCOVER FILES ----
ckpt_paths = sorted(glob.glob(os.path.join(MODELS_DIR, '*.pth')))
all_pngs   = sorted(glob.glob(os.path.join(TEST_DIR, '*.png')))
if MAX_PREVIEW is not None:
    all_pngs = all_pngs[:MAX_PREVIEW]

print(f'Found {len(ckpt_paths)} checkpoints in {MODELS_DIR}')
print(f'Found {len(all_pngs)} test PNGs in {TEST_DIR}')

# sanity checks
assert len(ckpt_paths) > 0, "No .pth files found."
assert len(all_pngs) > 0,   "No test PNGs found."


Found 24 checkpoints in /zpool/vladlab/active_drive/omaltz/scripts/geogaze/v3/resnet100/weights
Found 32 test PNGs in /zpool/vladlab/active_drive/omaltz/scripts/geogaze/stimuli/out/test_stimuli/test_pairs


In [8]:
# Purpose: Define your encoder+1x1 decoder and a helper to load any checkpoint.

class OneLayerDecoder(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, 1, kernel_size=1)

    def forward(self, feat, out_hw):
        logits = self.proj(feat)
        return F.interpolate(logits, size=out_hw, mode='bilinear', align_corners=False)

class EncoderHead(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.encoder = timm.create_model(backbone, pretrained=True, features_only=True, out_indices=(-1,))
        in_ch = self.encoder.feature_info[-1]['num_chs']
        self.head = OneLayerDecoder(in_ch)

    def forward(self, x):
        feat = self.encoder(x)[0]
        H, W = x.shape[-2:]
        return self.head(feat, (H, W))

def load_model(ckpt_path: str) -> nn.Module:
    model = EncoderHead(ARCH).to(device)
    for p in model.encoder.parameters():
        p.requires_grad = False
    model.encoder.eval()

    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
    model.load_state_dict(state, strict=True)
    model.eval()
    return model

############################
# After defining load_model(...)

def _print_backbone_info(m):
    # channels the decoder sees (should be 2048 for resnet50)
    in_ch = m.encoder.feature_info[-1]['num_chs']
    print(f"[INFO] ARCH={ARCH}  in_ch={in_ch}")

def debug_one(model, img_path):
    t = preprocess_image(img_path)             # your function from Cell 3
    with torch.no_grad():
        logits = model(t)                      # (1,1,H,W)
        probs  = torch.sigmoid(logits)[0,0]    # (H,W)
    print("logits  min/max/mean:", float(logits.min()), float(logits.max()), float(logits.mean()))
    print("probs   min/max/mean:", float(probs.min()),  float(probs.max()),  float(probs.mean()))
    print("head.proj.weight L2 :", float(model.head.proj.weight.norm()))
    print("head.proj.bias mean :", float(model.head.proj.bias.mean()))
    return probs.cpu().numpy()



In [5]:
# # ---- SINGLE CHECKPOINT DEBUG (run this once) ----
# # pick the first resnet checkpoint + first image
# dbg_ckpt = ckpt_paths[0]
# dbg_img  = all_pngs[0]

# model = load_model(dbg_ckpt)

# # channels the decoder sees (should be 2048 for resnet50)
# in_ch = model.encoder.feature_info[-1]['num_chs']
# print(f"[INFO] ARCH={ARCH}  in_ch={in_ch}  ckpt={Path(dbg_ckpt).name}")

# # forward once and inspect ranges
# t = preprocess_image(dbg_img)
# with torch.no_grad():
#     logits = model(t)                 # (1,1,H,W)
#     probs  = torch.sigmoid(logits)[0,0].cpu().numpy()

# print("logits min/max/mean:", float(logits.min()), float(logits.max()), float(logits.mean()))
# print("probs  min/max/mean:", float(probs.min()),  float(probs.max()),  float(probs.mean()))
# print("head.proj weight L2:", float(model.head.proj.weight.norm().cpu()))
# print("head.proj bias mean:", float(model.head.proj.bias.mean().cpu()))

# # optional: write quick debug images
# from imageio.v2 import imwrite
# from pathlib import Path
# dbg_dir = Path(OUT_ROOT) / "_debug"
# dbg_dir.mkdir(parents=True, exist_ok=True)
# imwrite(dbg_dir / "probs.png", (probs * 255).astype("uint8"))

# # try an adaptive threshold once to see if it's just thresholding
# import numpy as np
# thr_auto = float(np.quantile(probs, 0.85))
# print("auto threshold (85th %):", thr_auto)
# imwrite(dbg_dir / "mask_auto.png", (probs > thr_auto).astype("uint8") * 255)


In [9]:
# Purpose: Run every checkpoint over all test images and save masks.
import imageio.v2 as imageio

# parent output
OUT_ROOT = Path(OUT_ROOT)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def safe_name(s: str) -> str:
    """Make a filename-safe tag from the model name."""
    import re
    s = s.replace(os.sep, "_")
    return re.sub(r"[^A-Za-z0-9._+-]", "_", s)

@torch.no_grad()
def preprocess_image(path: str):
    img = Image.open(path).convert('RGB')
    arr = np.asarray(img, dtype=np.float32) / 255.0
    t = torch.from_numpy(arr).permute(2, 0, 1)
    t = (t - IMAGENET_MEAN) / IMAGENET_STD
    t = t.unsqueeze(0).to(device)  # (1,3,H,W) on device
    return t

@torch.no_grad()
def predict_mask_tensor(model: nn.Module, x: torch.Tensor, threshold: float = THRESHOLD) -> np.ndarray:
    logits = model(x)                                # (1,1,H,W) logits
    prob = torch.sigmoid(logits)[0, 0].cpu().numpy() # (H,W) float [0,1]
    pred = (prob > threshold).astype(np.uint8)       # 0/1
    return pred

for i, ckpt_path in enumerate(ckpt_paths, 1):
    model_name = Path(ckpt_path).stem
    model_tag  = safe_name(model_name)
    save_dir   = OUT_ROOT / model_name              # per-model subfolder
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f"[{i}/{len(ckpt_paths)}] {model_name}: running {len(all_pngs)} images...")
    model = load_model(ckpt_path)

    for p in all_pngs:
        x = preprocess_image(p)
        pred01 = predict_mask_tensor(model, x, threshold=THRESHOLD)

        # filename embeds the model name; mask pixels remain pure 0/255
        out_name = f"{Path(p).stem}__model={model_tag}_mask.png"
        out_path = save_dir / out_name
        imageio.imwrite(out_path, (pred01 * 255).astype(np.uint8))

print("Done.")


[1/24] resnet50.a1_in1k_maskL_bc_bs_best: running 32 images...
[2/24] resnet50.a1_in1k_maskL_bc_gc_best: running 32 images...
[3/24] resnet50.a1_in1k_maskL_bc_gs_best: running 32 images...
[4/24] resnet50.a1_in1k_maskL_bs_bc_best: running 32 images...
[5/24] resnet50.a1_in1k_maskL_bs_gc_best: running 32 images...
[6/24] resnet50.a1_in1k_maskL_bs_gs_best: running 32 images...
[7/24] resnet50.a1_in1k_maskL_gc_bc_best: running 32 images...
[8/24] resnet50.a1_in1k_maskL_gc_bs_best: running 32 images...
[9/24] resnet50.a1_in1k_maskL_gc_gs_best: running 32 images...
[10/24] resnet50.a1_in1k_maskL_gs_bc_best: running 32 images...
[11/24] resnet50.a1_in1k_maskL_gs_bs_best: running 32 images...
[12/24] resnet50.a1_in1k_maskL_gs_gc_best: running 32 images...
[13/24] resnet50.a1_in1k_maskR_bc_bs_best: running 32 images...
[14/24] resnet50.a1_in1k_maskR_bc_gc_best: running 32 images...
[15/24] resnet50.a1_in1k_maskR_bc_gs_best: running 32 images...
[16/24] resnet50.a1_in1k_maskR_bs_bc_best: runnin